In [0]:
data = [
    (101, "Alice", "HR", 5000.0, "2021-01-15", None),
    (102, "Bob", None, None, "2021-03-20", "New York"),
    (103, "Charlie", "IT", 7000.0, None, "Los Angeles"),
    (104, None, "Finance", None, "2021-07-10", None),
    (105, "Eve", None, 4500.0, "2021-09-01", "Chicago"),
    (106, None, "HR", None, None, None)
]
schema=['id','name','dept','salary','date','country']

df=spark.createDataFrame(data,schema)
df.show()

In [0]:
from pyspark.sql.functions import mean
mean_sal=df.select(mean('salary')).collect()[0][0]
print(mean_sal)
df.na.fill(mean_sal,subset=['salary']).show()

In [0]:
df.na.fill('unknown',subset=['dept']).show()

In [0]:

data = [
    (1, '2024-10-01'),
    (1, '2024-10-02'),
    (1, '2024-10-04'),
    (2, '2024-10-03'),
    (2, '2024-10-05'),
    (3, '2024-10-01'),
    (3, '2024-10-02'),
    (3, '2024-10-03'),
]


df = spark.createDataFrame(data, ["customer_id", "order_date"])
df.show()

df.createOrReplaceTempView("orig_tb")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

winspec=Window.partitionBy('customer_id').orderBy('order_date')
df_lag=df.withColumn('previous_order',lag('order_date').over(winspec))
df_lag.show()



In [0]:
df_pre=df_lag.withColumn('datdif',datediff('order_date','previous_order')).filter(col('datdif')==1)
df_pre.show()

In [0]:
df_final=df_pre.select('customer_id').distinct()
df_final.show()

In [0]:
%sql

SELECT DISTINCT customer_id
FROM (
    SELECT customer_id,
           order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_dt
    FROM orig_tb
) t
WHERE datediff(order_date, prev_dt) = 1;


In [0]:
data = [(1,'Abbot'),(2,'Doris'),(3,'Emerson'),(4,'Green'),(5,'Jeames')]
schema = ['id', 'student']


df = spark.createDataFrame(data = data, schema=schema)
df.show()

df_raw=df.createOrReplaceTempView("rawdata")
spark.sql("select * from rawdata").show()

In [0]:
spark.sql("select id, student, lag(student) over(order by id) as previous_val, lead(student) over(order by id) as next_val from rawdata  ").show()

spark.sql("SELECT id, student, MAX(id) OVER () AS max_id FROM rawdata").show()



In [0]:

spark.sql("""
WITH t AS (
  SELECT
    id,
    student,
    MAX(id) OVER () AS max_id
  FROM rawdata
)
SELECT
  id,
  CASE
    WHEN id % 2 = 1 AND id < max_id THEN LEAD(student) OVER(ORDER BY id)
    WHEN id % 2 = 0 THEN LAG(student) OVER(ORDER BY id)
    ELSE student
  END AS student
FROM t """).show()

In [0]:
data = [(101,'02-01-2024','N'),
(101,'03-01-2024','Y'),
(101,'04-01-2024','N'),
(101,'07-01-2024','Y'),
(102,'01-01-2024','N'),
(102,'02-01-2024','Y'),
(102,'03-01-2024','Y'),
(102,'04-01-2024','N'),
(102,'05-01-2024','Y'),
(102,'06-01-2024','Y'),
(102,'07-01-2024','Y'),
(103,'01-01-2024','N'),
(103,'04-01-2024','N'),
(103,'05-01-2024','Y'),
(103,'06-01-2024','Y'),
(103,'07-01-2024','N')
]
schema = ["emp_id" , "log_date" , "flag"]


df = spark.createDataFrame(data = data , schema = schema)
df.show()

df_rawdata=df.createOrReplaceTempView("rawdata")
spark.sql("select * from rawdata").show()
from pyspark.sql.functions import *

spark.sql("""select emp_id,log_date,TO_DATE(log_date, 'dd-MM-yyyy') AS log_dt from rawdata""").show()


In [0]:
from pyspark.sql import functions as F

spark.sql("""
 with base as (
	select emp_id,TO_DATE(log_date,'dd-MM-yyyy') as dt, flag from rawdata where flag='Y'
),
y_only as (
	select emp_id,dt,flag,ROW_NUMBER() over(PARTITION by emp_id order by dt) as rmp, dayofmonth(dt) as dat_part from base
), las_only as (
    select emp_id,dt,flag,rmp,dat_part,int(dat_part) - rmp as diff from y_only
)
    select emp_id,
    date_format(min(dt),'dd-MM-yyyy') AS start_date,
  	date_format(max(dt),'dd-MM-yyyy') AS end_date,
    count(*) AS consecutive_days from las_only group by emp_id,diff having count(*) > 1
""").show()


In [0]:
data = [
    (1, "Arjun Patel", "Male", 30, 60000, "Aarav Sharma"),
    (2, "Aarav Sharma", "Male", 28, 55000, "Zara Singh"),
    (3, "Zara Singh", "Female", 35, 70000, "Arjun Patel"),
    (4, "Priya Reddy", "Female", 32, 65000, "Aarav Sharma"),
    (1, "Arjun Patel", "Male", 30, 60000, "Aarav Sharma"),
    (6, "Naina Verma", "Female", 31, 72000, "Arjun Patel"),
    (1, "Arjun Patel", "Male", 30, 60000, "Aarav Sharma"),
    (4, "Priya Reddy", "Female", 32, 65000, "Aarav Sharma"),
    (5, "Aditya Kapoor", "Male", 28, 58000, "Zara Singh"),
    (10, "Anaya Joshi", "Female", 27, 59000, "Aarav Sharma"),
    (11, "Rohan Malhotra", "Male", 36, 73000, "Zara Singh"),
    (3, "Zara Singh", "Female", 35, 70000, "Arjun Patel")
]


df = spark.createDataFrame(data, schema=schema)
df.show()